# Python 项目与 uv

## Python 项目为什么需要专门管理

| 层次 | 要解决的问题 | 常见载体 |
| --- | --- | --- |
| Python 解释器 | 用哪个 Python 运行 | 系统 Python、uv 管理的 Python、`.python-version` |
| 项目元数据 | 项目叫什么、支持哪些 Python 版本 | `pyproject.toml` |
| 直接依赖 | 项目主动使用哪些包 | `[project].dependencies` |
| 间接依赖 | 直接依赖又需要哪些包 | 由解析器计算 |
| 精确版本 | 团队实际使用哪一组可兼容版本 | `uv.lock` |
| 隔离环境 | 把本项目的包与其他项目分开 | `.venv` |
| 命令执行 | 确保命令使用本项目环境 | `uv run` |

传统工具链可能分别使用 `pyenv`、`venv`、`pip`、`pip-tools`、`pipx` 和构建工具。uv 把这些高频工作统一到一个命令行工具中。

> uv 管理的是 Python 和 Python 项目，不会让 Python 代码本身运行得更快。它加速的是版本查找、依赖解析、下载、安装和环境同步。

## uv 是什么

uv 是用 Rust 编写的 Python 项目与包管理工具。它既能管理完整项目，也提供与 pip 工作流兼容的接口，还能管理 Python 版本和独立命令行工具。

| 使用场景 | 主要命令 | 是否修改项目声明 |
| --- | --- | --- |
| 创建和维护项目 | `uv init`、`uv add`、`uv remove`、`uv sync`、`uv lock` | 是 |
| 在项目环境中运行命令 | `uv run` | 可能自动更新锁文件和环境 |
| 管理 Python 版本 | `uv python` | `uv python pin` 会写 `.python-version` |
| 创建虚拟环境 | `uv venv` | 创建环境，不声明项目依赖 |
| 兼容 requirements/pip 工作流 | `uv pip` | 不自动修改 `pyproject.toml` |
| 运行或安装独立工具 | `uvx`、`uv tool` | 否，与项目环境隔离 |
| 构建和发布包 | `uv build`、`uv publish` | 读取项目构建配置 |

最重要的边界是：**新项目优先使用项目接口，旧项目迁移时才优先考虑 `uv pip` 接口**。`uv add httpx` 会维护项目声明和锁文件；`uv pip install httpx` 只修改某个环境，不能替代项目依赖声明。

## uv 为什么快

uv 的速度来自整体实现方式，不只是下载更快：

- 核心使用 Rust 实现，减少工具本身的运行开销。
- 依赖解析、下载和安装可以并行执行。
- 下载过的包、构建产物和 Git 依赖放入全局缓存，不必为每个项目重复下载。
- 在文件系统允许时，从缓存向虚拟环境硬链接或克隆文件，减少复制。
- 锁文件已经确定版本后，`uv sync` 不需要重新猜测依赖组合。

```mermaid
flowchart LR
    A["包索引或 Git 仓库"] --> B["uv 全局缓存"]
    B --> C["项目 A 的 .venv"]
    B --> D["项目 B 的 .venv"]
    E["uv.lock"] --> C
    F["另一个 uv.lock"] --> D
```

缓存是可重建的性能数据，不是项目真相。删除缓存不会删除 `pyproject.toml`、`uv.lock` 或源代码，下次只会重新下载和构建。

## 安装与查看帮助

Windows 可以使用官方安装脚本：

```powershell
powershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"
```

macOS 和 Linux：

```bash
curl -LsSf https://astral.sh/uv/install.sh | sh
```

安装后重新打开终端并确认：

```bash
uv --version
uv help
uv help sync
```

排查团队环境差异时，先比较 `uv --version`，再查看当前版本自带的帮助。

## 创建项目

`uv init` 创建项目骨架。日常开发应用可以直接执行：

```bash
uv init inventory-api --python 3.12
cd inventory-api
```

在已经存在的目录中，只需要创建最小项目配置时使用：

```bash
uv init --bare
```

典型项目会包含：

```text
inventory-api/
├── .python-version
├── .venv/
├── pyproject.toml
├── uv.lock
├── README.md
├── 应用源码
└── tests/
```

`.venv` 和 `uv.lock` 通常在首次解析或同步依赖时生成。

## 核心工作流与文件关系

uv 项目的核心不是某一条命令，而是声明、解析、落地、运行四个阶段：

```mermaid
flowchart LR
    A[".python-version<br/>本地解释器选择"] --> D["uv"]
    B["pyproject.toml<br/>项目与依赖范围"] --> D
    D --> C["uv.lock<br/>精确依赖方案"]
    C --> E[".venv<br/>本机安装结果"]
    E --> F["uv run<br/>在项目环境中执行"]
```

| 文件或目录 | 是否提交 Git | 能否手改 | 作用 |
| --- | --- | --- | --- |
| `pyproject.toml` | 是 | 是 | 项目声明和允许的依赖范围 |
| `uv.lock` | 通常是 | 不建议 | uv 计算出的精确、可复现依赖方案 |
| `.python-version` | 团队统一版本时提交 | 可以 | 告诉 uv 默认选择哪个 Python |
| `.venv/` | 否 | 不应手改 | 当前电脑上可直接运行的隔离环境 |
| uv 全局缓存 | 否 | 不应手改 | 跨项目复用下载和构建产物 |

删除 `.venv` 后执行 `uv sync` 可以重建环境；删除 `uv.lock` 会失去已经确定的版本方案，重新锁定时可能得到更新版本。两者的可恢复程度不同。

## 管理 Python 版本

项目能否运行首先取决于解释器。uv 可以下载并为项目固定 Python 版本：

```bash
uv python install 3.12
uv python pin 3.12
uv run python -V
```

`uv python pin 3.12` 会写入 `.python-version`。之后执行 `uv run` 或 `uv sync` 时，uv 会优先选择这个版本。

| 声明 | 回答的问题 | 示例 |
| --- | --- | --- |
| `.python-version` | 当前开发目录默认用哪个 Python | `3.12` |
| `project.requires-python` | 项目允许在哪些 Python 版本上运行 | `>=3.12,<3.14` |

团队应让这两处声明保持一致，避免本地解释器不在项目支持范围内。

## 虚拟环境

虚拟环境是一个独立的 Python 安装目录，保存本项目使用的解释器入口和第三方包。它防止项目 A 升级某个包时破坏项目 B。

uv 项目首次执行 `uv sync` 或 `uv run` 时，通常会自动创建项目根目录下的 `.venv`。也可以显式创建：

```bash
uv venv --python 3.12
uv sync
```

使用 `uv run` 时不需要激活环境：

```bash
uv run python -V
uv run python main.py
```

如果要让当前终端直接使用 `.venv`，Windows PowerShell 执行：

```powershell
.\.venv\Scripts\Activate.ps1
```

macOS 和 Linux 执行：

```bash
source .venv/bin/activate
```

激活只是临时修改当前终端的 `PATH`，不是安装依赖，也不会改变项目声明。对自动化脚本和团队文档，`uv run` 比“先激活环境再运行”更明确。

## `pyproject.toml`

`pyproject.toml` 是现代 Python 项目的标准配置入口，记录项目名称、Python 版本要求和直接依赖。

```toml
[project]
name = "inventory-api"
version = "0.1.0"
requires-python = ">=3.12,<3.14"
dependencies = [
    "fastapi>=0.115",
    "pydantic>=2",
]

[dependency-groups]
dev = [
    "pytest>=8",
]
```

优先使用 `uv add` 和 `uv remove` 修改依赖，因为它们会同时维护 TOML、锁文件和环境。手动编辑依赖后，应执行 `uv lock` 或 `uv sync` 重新同步。

## 添加、删除与升级依赖

添加和删除运行时依赖：

```bash
uv add fastapi
uv add "httpx>=0.28,<1"
uv remove httpx
```

测试、格式检查等只在开发时使用的工具放进开发依赖：

```bash
uv add --dev pytest ruff
```

`uv add` 默认会修改 `pyproject.toml`、更新 `uv.lock` 并同步项目环境。

升级依赖：

```bash
uv lock --upgrade
uv lock --upgrade-package httpx
```

升级仍受 `pyproject.toml` 中版本范围限制。需要接受新的主版本时，应先判断兼容性，再调整约束。

## 依赖解析与版本约束

项目只直接声明自己使用的包，但这些包还会依赖其他包。uv 的解析器要找到一组同时满足 Python 版本、操作系统、CPU 平台和所有版本约束的依赖。

```mermaid
flowchart TD
    A["项目"] --> B["FastAPI"]
    A --> C["HTTPX"]
    B --> D["Starlette"]
    B --> E["Pydantic"]
    C --> F["httpcore"]
```

| 约束 | 含义 | 适用判断 |
| --- | --- | --- |
| `package` | 允许解析器选择兼容版本 | 应用最终由锁文件固定 |
| `package>=2` | 至少需要某项新能力 | 常用的下界表达 |
| `package>=2,<3` | 接受 2.x，不接受 3.x | 已知主版本不兼容时使用 |
| `package==2.4.1` | 声明层也只允许一个版本 | 仅在确有严格要求时使用 |

`pyproject.toml` 的范围和 `uv.lock` 的精确版本不是重复：前者表达“允许什么”，后者记录“这次选了什么”。应用依靠锁文件复现；库还要给下游使用者留下合理的版本选择空间。

如果 A 要求 `pydantic<2`，B 要求 `pydantic>=2`，同一个环境无法同时满足，解析器会报告冲突。正确处理方式是升级或替换冲突依赖、调整确实过严的约束，不能靠反复重装碰运气。

## `uv.lock`

`uv.lock` 是 uv 根据项目声明计算出的精确依赖图，记录直接与间接依赖的实际版本和来源。

```bash
uv lock
uv lock --upgrade
uv lock --upgrade-package httpx
```

应用项目通常应提交 `uv.lock`，否则不同电脑和部署时间可能选择不同版本。不要手工编辑锁文件；要改变版本，应修改项目声明或使用升级命令，让 uv 重新计算。

## `uv sync`

`uv sync` 根据项目声明和锁文件创建或更新 `.venv`：安装缺少的包、调整版本，并删除不属于项目依赖的多余包。

```bash
uv sync
uv sync --locked
uv sync --no-dev
```

| 命令 | 适合场景 |
| --- | --- |
| `uv sync` | 日常开发，必要时更新锁文件 |
| `uv sync --locked` | CI 或部署，锁文件过期时直接报错 |
| `uv sync --no-dev` | 不需要测试和格式化工具的运行环境 |

删除 `.venv` 后重新执行 `uv sync`，可以按锁文件重建环境。

## `uv run`

`uv run` 在当前项目环境中执行命令。它会找到项目并确保所需环境可用，因此通常不需要先激活虚拟环境。

```bash
uv run python main.py
uv run pytest
uv run jupyter lab
```

直接执行 `python` 或 `pytest` 可能命中系统路径；`uv run` 明确把命令放在当前项目上下文中，是日常运行项目和工具的首选方式。

## 开发依赖

项目运行所需的包属于运行时依赖，测试和代码检查工具属于开发依赖。

```bash
uv add fastapi
uv add --dev pytest ruff
uv sync
uv run pytest
uv run ruff check .
```

uv 默认会安装 `dev` 依赖组，方便本地开发；只运行应用且不需要测试工具时，可以使用 `uv sync --no-dev`。

## 克隆并运行已有项目

一个提交规范的 uv 项目应该让新成员不需要手工逐个安装依赖：

```bash
git clone https://github.com/example/inventory-api.git
cd inventory-api
uv sync --locked
uv run pytest
uv run python -m app
```

Git 中通常提交：

- `pyproject.toml`：项目声明。
- `uv.lock`：精确依赖方案。
- `.python-version`：团队统一的 Python 版本。
- 源代码、测试和配置模板。

通常忽略 `.venv/`、构建产物和包含密钥的 `.env`。

## 排查问题

| 现象 | 先检查什么 | 常用处理 |
| --- | --- | --- |
| Python 版本不对 | uv 实际选择了哪个解释器 | `uv run python -V`，检查 `.python-version` |
| 导入包失败 | 依赖是否已同步、命令是否在项目环境运行 | `uv sync` 后使用 `uv run` |
| 锁文件过期 | `pyproject.toml` 是否变更 | 执行 `uv lock` 并提交 `uv.lock` |
| 依赖解析冲突 | 哪些版本范围无法同时满足 | 阅读错误中的依赖链，升级或调整约束 |
| 本地包被同步删除 | 是否绕过项目声明手工安装 | 使用 `uv add` 正式声明依赖 |

需要查看更多过程时可以执行 `uv -v sync`。遇到网络问题先检查代理、DNS 和证书，不要轻易跳过 TLS 校验。

## 新手常见误区

| 误区 | 实际情况 |
| --- | --- |
| uv 只是更快的 pip | uv 还管理 Python、项目声明、锁文件、虚拟环境和命令运行 |
| 有 `uv.lock` 就不需要 `pyproject.toml` | TOML 表达项目意图，锁文件记录解析出的精确版本 |
| `.venv` 应提交 Git | 它与本机环境相关，应由 `uv sync` 重建 |
| 激活环境等于同步依赖 | 激活只修改当前终端的 `PATH`，不会安装依赖 |
| 手工安装包就完成了依赖管理 | 正式依赖应使用 `uv add` 写入项目声明 |
| 删除缓存或 `.venv` 会破坏项目 | 只要源码、TOML 和锁文件还在，都可以重建 |
| uv 会让业务代码执行更快 | uv 优化的是解释器和依赖管理流程，不改变 Python 运行性能 |

**面试提示**：Python 工程常考虚拟环境解决什么、直接依赖与间接依赖的区别，以及 `pyproject.toml` 与锁文件为什么同时存在。